# OGM-G
This notebook applies the systematic Lyapunov-function discovery procedure to the 
OGM-G algorithm, introduced in "Optimizing the Efficiency of First-Order Methods for
Decreasing the Gradient of Smooth Convex Functions" by Kim and Fessler (2021).
This algorithm reduces the gradient norm $\|\nabla f(x_N)\|^2$ in $O(1/N^2)$-order
with respect to the initial function-value condition $f(x_0) - f_\star \le \frac12 LR^2$.

## Import the required libraries

In [1]:
import functools
import itertools

import numpy as np
import pepflow as pf
import sympy as sp
from IPython.display import Math, display

from pepflow.lyapunov_utils import (
    find_symmetric_coefficient_matrix,
    select_independent_subset,
    vectors_in_column_space,
)

## Define the function class

In [2]:
L = pf.Parameter("L")
f = pf.SmoothConvexFunction(is_basis=True, tags=["f"], L=L)

## Write a function that generates a PEPContext object associated with OGM-G

### We use the same $\theta_i$ sequence as in OGM, but its reversed version: $\psi_i=\theta_{N-i}$

In [3]:
@functools.cache
def theta(i: int, N: int):
    if i == -1:
        return sp.S(0)
    if i == N:
        return sp.S(1) / 2 * (sp.S(1) + sp.sqrt(8 * theta(N - 1, N) ** 2 + sp.S(1)))
    return sp.S(1) / 2 * (sp.S(1) + sp.sqrt(4 * theta(i - 1, N) ** 2 + sp.S(1)))


def reverse_theta(i: int, N: int):
    return theta(N - i, N)


def make_reversed_theta_parameters(N: int):
    return [pf.Parameter(f"reversed_theta_{i}") for i in range(N + 2)]


def reversed_theta_resolve(N: int):
    return {
        f"reversed_theta_{i}": np.float64(reverse_theta(i, N)) for i in range(N + 2)
    }


def reversed_theta_symbolic_resolve(N: int):
    return {f"reversed_theta_{i}": reverse_theta(i, N) for i in range(N + 2)}

In [4]:
def make_ctx_ogm_g(
    ctx_name: str,
    N: int | sp.Integer,
    stepsize: pf.Parameter,
    reversed_theta: list[pf.Parameter],
) -> pf.PEPContext:
    N = int(N)
    ctx_ogm_g = pf.PEPContext(ctx_name).set_as_current()
    x = pf.Vector(is_basis=True, tags=["x_0"])
    z = x
    z = z - stepsize * (reversed_theta[0] + 1) / 2 * f.grad(x)
    z.add_tag("z_1")
    f.set_stationary_point("x_star")

    for i in range(1, N + 1):
        y = x - stepsize * f.grad(x)
        x = (reversed_theta[i + 1] / reversed_theta[i]) ** 4 * y + (
            1 - (reversed_theta[i + 1] / reversed_theta[i]) ** 4
        ) * z
        y.add_tag(f"y_{i}")
        x.add_tag(f"x_{i}")

        z = z - stepsize * reversed_theta[i] * f.grad(x)
        z.add_tag(f"z_{i + 1}")

    return ctx_ogm_g

## Numerical PEP Solve

### We first solve the dense PEP and read off the interpolation certificate. OGM-G has no Gram-matrix slack in this certificate, so the sparse interpolation multipliers carry the proof.

In [5]:
N = 5
N_int = int(N)
R = pf.Parameter("R")
L_value = np.float64(1)
R_value = np.float64(1)

reversed_theta = make_reversed_theta_parameters(N_int)
resolve_prf = {"L": L_value, "R": R_value}
resolve_prf.update(reversed_theta_resolve(N_int))

ctx_prf = make_ctx_ogm_g(
    ctx_name="ctx_ogm_g_prf",
    N=N_int,
    stepsize=1 / L,
    reversed_theta=reversed_theta,
)
pb_prf = pf.PEPBuilder(ctx_prf)
pb_prf.add_initial_constraint(
    (f(ctx_prf["x_0"]) - f(ctx_prf["x_star"])).le(R, name="initial_condition")
)
pb_prf.set_performance_metric((f.grad(ctx_prf[f"x_{N_int}"])) ** 2)

result_dense = pb_prf.solve(resolve_parameters=resolve_prf)
lamb_dense = result_dense.get_scalar_constraint_dual_value_in_numpy(f)

print("dense optimum:", result_dense.opt_value)

dense optimum: 0.0743591042068431


In [6]:
pf.pprint_labeled_matrix(lamb_dense.matrix, lamb_dense.row_names, lamb_dense.col_names)

<IPython.core.display.Math object>

### The closed-form multipliers match the dense numerical certificate.

In [7]:
def tag_to_index(tag: str, N: int | sp.Integer = N_int) -> int:
    idx = tag.split("_", 1)[1]
    if idx.isdigit():
        return int(idx)
    if idx == "star":
        return int(N) + 1
    raise ValueError(f"Unknown point tag: {tag}")


def lambda_formula(
    tag_i: str,
    tag_j: str,
    N: int | sp.Integer = N_int,
    reverse_theta_fn=reverse_theta,
):
    N = int(N)
    i = tag_to_index(tag_i, N)
    j = tag_to_index(tag_j, N)

    if i == N:
        if j == 0:
            return 1 / reverse_theta_fn(1, N) ** 2 - 2 / reverse_theta_fn(0, N) ** 2
        if j < N:
            return 1 / reverse_theta_fn(j + 1, N) ** 2 - 1 / reverse_theta_fn(j, N) ** 2
        if j == N:
            return 1 - lambda_formula(
                f"x_{N - 1}", f"x_{N}", N=N, reverse_theta_fn=reverse_theta_fn
            )
        if j == N + 1:
            return 2 / reverse_theta_fn(0, N) ** 2

    if i < N and i + 1 == j:
        return 1 / reverse_theta_fn(i + 1, N) ** 2

    return 0


def lambda_numeric(tag_i: str, tag_j: str, N: int | sp.Integer = N_int):
    return np.float64(lambda_formula(tag_i, tag_j, N=N))


lamb_cand = pf.pprint_labeled_matrix(
    lambda tag_i, tag_j: lambda_formula(tag_i, tag_j, N_int),
    lamb_dense.row_names,
    lamb_dense.col_names,
    return_matrix=True,
)
lamb_cand_numeric = np.vectorize(lambda x: float(sp.N(x)))(lamb_cand)

print(
    "closed-form lambdas match dense certificate?",
    np.allclose(lamb_cand_numeric, lamb_dense.matrix, atol=1e-4),
)

<IPython.core.display.Math object>

closed-form lambdas match dense certificate? True


### We now enforce the observed zero pattern and solve the reduced PEP.

In [8]:
relaxed_constraints = []
for tag_i in lamb_dense.row_names:
    for tag_j in lamb_dense.col_names:
        if abs(lambda_numeric(tag_i, tag_j, N=N_int)) > 1e-12:
            continue
        relaxed_constraints.append(f"f:{tag_i},{tag_j}")

pb_prf.set_relaxed_constraints(relaxed_constraints)
result = pb_prf.solve(resolve_parameters=resolve_prf)

# Dual variable associated with the initial condition.
tau_sol = result.dual_var_manager.dual_value("initial_condition")
# Dual variables associated with the interpolation conditions of f.
lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(f)
# Dual matrix associated with the Gram matrix G.
S_sol = result.get_gram_dual_matrix()

print("sparse optimum:", result.opt_value)
print("tau:", tau_sol)

sparse optimum: 0.0743525459451049
tau: 0.07435254822048483


In [9]:
lamb_sol.pprint()

<IPython.core.display.Math object>

### The Gram dual matrix S is zero, so $|\mathcal{J}| = 0$

In [10]:
S_sol.pprint()

<IPython.core.display.Math object>

---

## Step 1. Propose a Candidate Lyapunov Function

### Let $\mathcal I_0=\{(N,0)\},\,\, \mathcal I_k=\{(j,j+1):j=0,\ldots,k-1\}\cup\{(N,j):j=0,\ldots,\min\{k,N-1\}\}$ for $k=1,\dots,N$ and $\mathcal J_k = \emptyset$ for $k=0,\dots,N$.
### In particular, we have $\mathcal I \setminus \mathcal I_N = \{(N,\star)\}$.

In [11]:
def named_matrix_entry(named_matrix, row_name: str, col_name: str):
    row = named_matrix.row_names.index(row_name)
    col = named_matrix.col_names.index(col_name)
    return named_matrix.matrix[row, col]


def lamb_numeric(tag_i: str, tag_j: str):
    return named_matrix_entry(lamb_sol, tag_i, tag_j)


V_raw = []
partial_sum = lamb_numeric(f"x_{N_int}", "x_0") * f.interp_ineq(f"x_{N_int}", "x_0")
V_raw.append(partial_sum)

for step in range(N_int):
    partial_sum += lamb_numeric(f"x_{step}", f"x_{step + 1}") * f.interp_ineq(
        f"x_{step}", f"x_{step + 1}"
    )
    partial_sum += lamb_numeric(f"x_{N_int}", f"x_{step + 1}") * f.interp_ineq(
        f"x_{N_int}", f"x_{step + 1}"
    )
    V_raw.append(partial_sum)

full_interp_sum = V_raw[-1] + lamb_numeric(f"x_{N_int}", "x_star") * f.interp_ineq(
    f"x_{N_int}", "x_star"
)

print("V_raw contains:", [f"V_{i}" for i in range(len(V_raw))])

V_raw contains: ['V_0', 'V_1', 'V_2', 'V_3', 'V_4', 'V_5']


## Step 2. Check for admissibility

### The rank seems to be 4 for generic $k=1,\dots,N-2$, while it drops to 3 when $k=0,N-1$ and to 1 when $k=N$
### The set of active $\mathbf{f}$-coordinates seems to be $\{(0,k,N)\}$

In [12]:
pm = pf.ExpressionManager(ctx_prf, resolve_parameters=resolve_prf)
scalar_labels = [str(s) for s in ctx_prf.basis_scalars()]

for i, V in enumerate(V_raw):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-5)
    func_coords = V_eval.func_coords.astype(float)
    support = [
        (scalar_labels[j], value)
        for j, value in enumerate(func_coords)
        if abs(value) > 1e-7
    ]
    print(f"V_{i}: rank={rank}, nonzero function coordinates={support}")

V_0: rank=3, nonzero function coordinates=[('f(x_0)', np.float64(0.01776044383336822)), ('f(x_5)', np.float64(-0.01776044383336822))]
V_1: rank=4, nonzero function coordinates=[('f(x_0)', np.float64(-0.07435254822051131)), ('f(x_1)', np.float64(0.13225147634012877)), ('f(x_5)', np.float64(-0.05789892811961744))]
V_2: rank=4, nonzero function coordinates=[('f(x_0)', np.float64(-0.07435254822051131)), ('f(x_2)', np.float64(0.20783276010498508)), ('f(x_5)', np.float64(-0.13348021188447626))]
V_3: rank=4, nonzero function coordinates=[('f(x_0)', np.float64(-0.07435254822051131)), ('f(x_3)', np.float64(0.38196601676825404)), ('f(x_5)', np.float64(-0.30761346854778693))]
V_4: rank=3, nonzero function coordinates=[('f(x_0)', np.float64(-0.07435254822051131)), ('f(x_4)', np.float64(1.000000000021469)), ('f(x_5)', np.float64(-0.925647451801092))]
V_5: rank=1, nonzero function coordinates=[('f(x_0)', np.float64(-0.07435254822051131)), ('f(x_5)', np.float64(0.07435254822032755))]


## Step 3. Build a set of special candidate vectors

### Include all $x_i, \nabla f(x_i), y_i, z_i$ and their pairwise differences

In [13]:
ctx_prf.set_as_current()
x = [ctx_prf[f"x_{i}"] for i in range(N_int + 1)]
y = [None] + [ctx_prf[f"y_{i}"] for i in range(1, N_int + 1)]
z = [None] + [ctx_prf[f"z_{i}"] for i in range(1, N_int + 2)]
x_star = ctx_prf["x_star"]
g = [f.grad(x_i) for x_i in x]

lyap_basis_candidate = []
seen_candidates = set()


def add_candidate(vector):
    key = str(vector)
    if key in seen_candidates:
        return
    seen_candidates.add(key)
    lyap_basis_candidate.append(vector)


for vector in ctx_prf.basis_vectors():
    add_candidate(vector)
for vector in x + y[1:] + z[1:] + [x_star] + g:
    add_candidate(vector)

base_count = len(lyap_basis_candidate)
for i, j in itertools.combinations(range(base_count), 2):
    add_candidate(lyap_basis_candidate[i] - lyap_basis_candidate[j])

print("number of candidate vectors:", len(lyap_basis_candidate))

number of candidate vectors: 300


## Step 4. Complete a Sparse Basis of $\mathbf V_i$

### We find that $\{\mathbf{g}_k, \mathbf{g}_N, \mathbf{x}_{k+1} - \mathbf{y}_{k+1}, \mathbf{z}_{k+1} - \mathbf{z}_{N+1}\}$ is a basis of $\mathcal{R}(\mathbf{V}_k)$ for $k=1,\dots,N-2$.

- This becomes linearly dependent and the rank collapses to 3 when $k=0$, but in the end, the same quadratic formula for $k=1,\dots,N-2$ can be used for this case.
- When $k=N-1$, the basis collapses to $\{\mathbf{g}_{N-1}, \mathbf{g}_N, \mathbf{x}_{N} - \mathbf{y}_{N}\}$.

In [14]:
def vector_rank(vectors, tol=1e-7):
    if not vectors:
        return 0
    coords = [pm.eval_vector(v).coords.astype(float).ravel() for v in vectors]
    return np.linalg.matrix_rank(np.stack(coords, axis=1), tol=tol)


def expected_basis_for_V(i: int):
    if i == 0:
        return [g[0], g[N_int], z[1] - z[N_int + 1]]
    if i <= N_int - 2:
        return [g[i], g[N_int], x[i + 1] - y[i + 1], z[i + 1] - z[N_int + 1]]
    if i == N_int - 1:
        return [g[N_int - 1], g[N_int], x[N_int] - y[N_int]]
    return [g[N_int]]


def complete_basis_by_sparse_search(V, fixed_vectors, expected_completion_label: str):
    aligned = vectors_in_column_space(
        V,
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=resolve_prf,
        rtol=1e-5,
        atol=1e-5,
    )
    target_rank = np.linalg.matrix_rank(
        pm.eval_scalar(V).inner_prod_coords.astype(float), tol=1e-5
    )

    best = None
    for candidate in aligned:
        trial_basis = fixed_vectors + [candidate]
        if vector_rank(trial_basis, tol=1e-7) != target_rank:
            continue
        try:
            C = find_symmetric_coefficient_matrix(
                V,
                trial_basis,
                pep_context=ctx_prf,
                resolve_parameters=resolve_prf,
                indep_tol=1e-7,
            )
        except ValueError:
            continue

        zero_count = int(np.sum(np.abs(C) < 1e-6))
        expected_match = int(str(candidate) == expected_completion_label)
        score = (zero_count, expected_match, -len(str(candidate)))
        if best is None or score > best[0]:
            best = (score, trial_basis, C, aligned)

    if best is None:
        raise RuntimeError("No completion vector found.")
    _, trial_basis, C, aligned = best
    return trial_basis, C, aligned

In [15]:
completion_records = {}
for i, V in enumerate(V_raw):
    aligned = vectors_in_column_space(
        V,
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=resolve_prf,
        rtol=1e-5,
        atol=1e-5,
    )
    independent_vectors, _ = select_independent_subset(
        aligned,
        ctx_prf,
        resolve_parameters=resolve_prf,
        tol=1e-5,
    )
    print(
        f"V_{i}: aligned candidates={len(aligned)}, "
        f"greedy independent size={len(independent_vectors)}"
    )
    pf.pprint_labeled_vector(
        pm.eval_scalar(V).func_coords.astype(float),
        scalar_labels,
    )

    if i == 0:
        dependent_family = [g[0], g[N_int], x[1] - y[1], z[1] - z[N_int + 1]]
        print(
            "rank of [grad_f(x_0), grad_f(x_N), p_0, v_0]:",
            vector_rank(dependent_family, tol=1e-7),
        )
        basis_i = expected_basis_for_V(i)
        C_i = find_symmetric_coefficient_matrix(
            V,
            basis_i,
            pep_context=ctx_prf,
            resolve_parameters=resolve_prf,
            indep_tol=1e-7,
        )
    elif i == N_int:
        basis_i = expected_basis_for_V(i)
        C_i = find_symmetric_coefficient_matrix(
            V,
            basis_i,
            pep_context=ctx_prf,
            resolve_parameters=resolve_prf,
            indep_tol=1e-7,
        )
    else:
        expected = expected_basis_for_V(i)
        basis_i, C_i, _ = complete_basis_by_sparse_search(
            V,
            expected[:-1],
            str(expected[-1]),
        )

    labels_i = [str(v) for v in basis_i]
    completion_records[i] = (basis_i, C_i)
    pf.pprint_labeled_matrix(np.round(C_i, 9), labels_i, labels_i)

V_0: aligned candidates=24, greedy independent size=3


<IPython.core.display.Math object>

rank of [grad_f(x_0), grad_f(x_N), p_0, v_0]: 3


<IPython.core.display.Math object>

V_1: aligned candidates=51, greedy independent size=4


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

V_5: aligned candidates=4, greedy independent size=1


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### We hypothesize a Lyapunov function 
$$
V_i=a_i\bigl(f(x_i)-f(x_N)\bigr)+b_i\bigl(f(x_0)-f(x_N)\bigr)+c_i\|g_i\|^2+d_i\|g_N\|^2+e_i\langle x_{i+1} - y_{i+1}, z_{i+1} - z_{N+1}\rangle
$$
### for $i=0,\dots,N-1$. The case $i=0$ is not a counterexample; the cross term is formed with $x_1-y_1$ and is linearly dependent with $\nabla f(x_0)$ because of the special initialization of $z_1$.

---

## Step 5. Analytic Proof in the Completed Basis

### Closed-form expressions for $\lambda$ certificates. In particular, $\lambda_{i,i+1}=\psi_{i+1}^{-2}$, $\lambda_{N,0}=\psi_1^{-2}-2\psi_0^{-2}$, $\lambda_{N,i}=\psi_{i+1}^{-2}-\psi_i^{-2}$ for $i=1,\dots,N-1$, and $\lambda_{N,\star}=2\psi_0^{-2}$.

In [16]:
def proof_tag_to_index(tag: str, N: int | sp.Integer = N_int) -> int:
    idx = tag.split("_", 1)[1]
    if idx.isdigit():
        return int(idx)
    if idx == "star":
        return int(N) + 1
    raise ValueError(f"Unknown point tag: {tag}")


def proof_lambda_formula(
    tag_i: str,
    tag_j: str,
    N: int | sp.Integer = N_int,
    reverse_theta_fn=reverse_theta,
):
    N = int(N)
    i = proof_tag_to_index(tag_i, N)
    j = proof_tag_to_index(tag_j, N)

    if i == N:
        if j == 0:
            return 1 / reverse_theta_fn(1, N) ** 2 - 2 / reverse_theta_fn(0, N) ** 2
        if j < N:
            return 1 / reverse_theta_fn(j + 1, N) ** 2 - 1 / reverse_theta_fn(j, N) ** 2
        if j == N:
            return 1 - proof_lambda_formula(
                f"x_{N - 1}", f"x_{N}", N=N, reverse_theta_fn=reverse_theta_fn
            )
        if j == N + 1:
            return 2 / reverse_theta_fn(0, N) ** 2

    if i < N and i + 1 == j:
        return 1 / reverse_theta_fn(i + 1, N) ** 2

    return 0


def lambda_numeric(tag_i: str, tag_j: str, N: int | sp.Integer = N_int):
    return np.float64(proof_lambda_formula(tag_i, tag_j, N=N))


lamb_cand = pf.pprint_labeled_matrix(
    lambda tag_i, tag_j: proof_lambda_formula(tag_i, tag_j, N_int),
    lamb_dense.row_names,
    lamb_dense.col_names,
    return_matrix=True,
)
lamb_cand_numeric = np.vectorize(lambda x: float(sp.N(x)))(lamb_cand)

print(
    "closed-form lambdas match dense certificate?",
    np.allclose(lamb_cand_numeric, lamb_dense.matrix, atol=1e-4),
)

<IPython.core.display.Math object>

closed-form lambdas match dense certificate? True


In [17]:
def simplify_sympy_matrix(array):
    return sp.Matrix(array).applyfunc(
        lambda entry: sp.factor(sp.together(sp.simplify(entry)))
    )


def reduce_by_quadratic_recurrence(expr, eliminated_symbol, recurrence_expr):
    expr = sp.factor(sp.together(sp.simplify(expr)))
    numerator, denominator = sp.fraction(expr)
    if numerator == 0:
        return sp.S(0)
    try:
        numerator_reduced = sp.rem(
            sp.Poly(sp.expand(numerator), eliminated_symbol),
            sp.Poly(recurrence_expr, eliminated_symbol),
        ).as_expr()
        return sp.factor(sp.together(sp.simplify(numerator_reduced / denominator)))
    except sp.PolynomialError:
        return expr


def symbolic_parts(expr, ctx, resolve_parameters):
    pm_check = pf.ExpressionManager(ctx, resolve_parameters=resolve_parameters)
    matrix_part = simplify_sympy_matrix(pm_check.eval_scalar(expr).inner_prod_coords)
    func_part = simplify_sympy_matrix(pm_check.eval_scalar(expr).func_coords)
    return matrix_part, func_part


def symbolic_vector_coords(vec, ctx, resolve_parameters):
    pm_check = pf.ExpressionManager(ctx, resolve_parameters=resolve_parameters)
    return simplify_sympy_matrix(pm_check.eval_vector(vec).coords)


def display_symbolic_residual(name, matrix_part, func_part):
    print(name)
    print("quadratic residual:")
    display(matrix_part)
    print("function-value residual:")
    display(func_part)
    print(
        "identity verified?",
        matrix_part == sp.zeros(*matrix_part.shape)
        and func_part == sp.zeros(*func_part.shape),
    )


def display_symbolic_vector_residual(name, coords):
    print(name)
    display(coords)
    print("identity verified?", coords == sp.zeros(*coords.shape))

### We solve $0=V_{i+1}-V_i-\lambda_{i,i+1} I(x_i,x_{i+1})-\lambda_{N,i+1} I(x_N,x_{i+1})$ for Lyapunov coefficients, $i=0,\dots,N-2$

In [18]:
ctx_ogmg_interior = pf.PEPContext("ogmg_sparse_interior_proof").set_as_current()

g_i_abs = pf.Vector(is_basis=True, tags=["g_i"])
g_ip1_abs = pf.Vector(is_basis=True, tags=["g_{i+1}"])
g_N_abs = pf.Vector(is_basis=True, tags=["g_N"])
p_i_abs = pf.Vector(is_basis=True, tags=["p_i"])
v_i_abs = pf.Vector(is_basis=True, tags=["v_i"])

f_0_abs = pf.Scalar(is_basis=True, tags=["f_0"])
f_i_abs = pf.Scalar(is_basis=True, tags=["f_i"])
f_ip1_abs = pf.Scalar(is_basis=True, tags=["f_{i+1}"])
f_N_abs = pf.Scalar(is_basis=True, tags=["f_N"])

psi_ip1_par = pf.Parameter("psi_{i+1}")
psi_ip2_par = pf.Parameter("psi_{i+2}")
psi_0_par = pf.Parameter("psi_0")

a_i_par = pf.Parameter("a_i")
a_ip1_par = pf.Parameter("a_{i+1}")
b_i_par = pf.Parameter("b_i")
b_ip1_par = pf.Parameter("b_{i+1}")
c_i_par = pf.Parameter("c_i")
c_ip1_par = pf.Parameter("c_{i+1}")
d_i_par = pf.Parameter("d_i")
d_ip1_par = pf.Parameter("d_{i+1}")
e_i_par = pf.Parameter("e_i")
e_ip1_par = pf.Parameter("e_{i+1}")

p_ip1_abs = (
    (2 * psi_ip2_par - 1)
    / psi_ip2_par**2
    * (
        (psi_ip1_par - 1) ** 2 / (2 * psi_ip1_par - 1) * p_i_abs
        - (psi_ip1_par - 1) / L * g_ip1_abs
    )
)
v_ip1_abs = v_i_abs - psi_ip1_par / L * g_ip1_abs

V_i_unknown = (
    a_i_par * (f_i_abs - f_N_abs)
    + b_i_par * (f_0_abs - f_N_abs)
    + c_i_par * g_i_abs**2
    + d_i_par * g_N_abs**2
    + e_i_par * (p_i_abs * v_i_abs)
)
V_ip1_unknown = (
    a_ip1_par * (f_ip1_abs - f_N_abs)
    + b_ip1_par * (f_0_abs - f_N_abs)
    + c_ip1_par * g_ip1_abs**2
    + d_ip1_par * g_N_abs**2
    + e_ip1_par * (p_ip1_abs * v_ip1_abs)
)

x_i_minus_x_ip1 = g_i_abs / L - p_i_abs
x_N_minus_x_ip1 = (
    (psi_ip1_par - 1) ** 2 / (2 * psi_ip1_par - 1) * p_i_abs - v_i_abs + g_N_abs / L
)

I_adj_abs = (
    f_ip1_abs
    - f_i_abs
    + g_ip1_abs * x_i_minus_x_ip1
    + sp.S(1) / (2 * L) * (g_i_abs - g_ip1_abs) ** 2
)
I_anchor_abs = (
    f_ip1_abs
    - f_N_abs
    + g_ip1_abs * x_N_minus_x_ip1
    + sp.S(1) / (2 * L) * (g_N_abs - g_ip1_abs) ** 2
)

interior_unknown_residual = (
    V_ip1_unknown
    - V_i_unknown
    - 1 / psi_ip1_par**2 * I_adj_abs
    - (1 / psi_ip2_par**2 - 1 / psi_ip1_par**2) * I_anchor_abs
)

In [19]:
L_sym = sp.Symbol("L")
psi_ip1_sym = sp.Symbol(r"\psi_{i+1}")
psi_ip2_sym = sp.Symbol(r"\psi_{i+2}")
psi_0_sym = sp.Symbol(r"\psi_0")

a_i_sym, a_ip1_sym, b_i_sym, b_ip1_sym = sp.symbols(r"a_i a_{i+1} b_i b_{i+1}")
c_i_sym, c_ip1_sym, d_i_sym, d_ip1_sym, e_i_sym, e_ip1_sym = sp.symbols(
    r"c_i c_{i+1} d_i d_{i+1} e_i e_{i+1}"
)

ogmg_interior_resolve = {
    "L": L_sym,
    "psi_{i+1}": psi_ip1_sym,
    "psi_{i+2}": psi_ip2_sym,
    "psi_0": psi_0_sym,
    "a_i": a_i_sym,
    "a_{i+1}": a_ip1_sym,
    "b_i": b_i_sym,
    "b_{i+1}": b_ip1_sym,
    "c_i": c_i_sym,
    "c_{i+1}": c_ip1_sym,
    "d_i": d_i_sym,
    "d_{i+1}": d_ip1_sym,
    "e_i": e_i_sym,
    "e_{i+1}": e_ip1_sym,
}

interior_matrix, interior_func = symbolic_parts(
    interior_unknown_residual,
    ctx_ogmg_interior,
    ogmg_interior_resolve,
)
recurrence_relation = psi_ip1_sym**2 - psi_ip1_sym - psi_ip2_sym**2


def reduce_ogmg(entry):
    return reduce_by_quadratic_recurrence(entry, psi_ip1_sym, recurrence_relation)


unknowns = (
    a_i_sym,
    a_ip1_sym,
    b_i_sym,
    b_ip1_sym,
    c_i_sym,
    c_ip1_sym,
    d_i_sym,
    d_ip1_sym,
    e_i_sym,
    e_ip1_sym,
)
equations = [
    reduce_ogmg(entry)
    for entry in list(interior_matrix) + list(interior_func)
    if any(entry.has(unknown) for unknown in unknowns)
]
equations = [eq for eq in equations if eq != 0]
coefficient_matrix, rhs = sp.linear_eq_to_matrix(equations, unknowns)
solution_tuple = next(iter(sp.linsolve((coefficient_matrix, rhs), unknowns)))
solved_solution = {
    unknown: sp.factor(sp.together(sp.simplify(expression)))
    for unknown, expression in zip(unknowns, solution_tuple)
}

print("coefficient matrix rank:", coefficient_matrix.rank())
print("coefficient nullity:", len(coefficient_matrix.nullspace()))
print("transition solution before boundary/terminal conditions fix the free constants:")
display(
    Math(
        r"\begin{aligned}"
        + r"\\".join(
            sp.latex(sp.Eq(unknown, solved_solution[unknown])) for unknown in unknowns
        )
        + r"\end{aligned}"
    )
)

closed_form_ogmg = {
    a_i_sym: 1 / psi_ip1_sym**2,
    a_ip1_sym: 1 / psi_ip2_sym**2,
    b_i_sym: -2 / psi_0_sym**2,
    b_ip1_sym: -2 / psi_0_sym**2,
    c_i_sym: -1 / (2 * L_sym * psi_ip1_sym**2),
    c_ip1_sym: -1 / (2 * L_sym * psi_ip2_sym**2),
    d_i_sym: (psi_0_sym**2 - 2 * psi_ip1_sym**2)
    / (2 * L_sym * psi_0_sym**2 * psi_ip1_sym**2),
    d_ip1_sym: (psi_0_sym**2 - 2 * psi_ip2_sym**2)
    / (2 * L_sym * psi_0_sym**2 * psi_ip2_sym**2),
    e_i_sym: L_sym / (psi_ip1_sym**2 * (2 * psi_ip1_sym - 1)),
    e_ip1_sym: L_sym / (psi_ip2_sym**2 * (2 * psi_ip2_sym - 1)),
}

print("closed-form coefficients selected by the boundary/terminal conditions:")
display(
    Math(
        r"\begin{aligned}"
        + r"\\".join(
            sp.latex(sp.Eq(unknown, closed_form_ogmg[unknown])) for unknown in unknowns
        )
        + r"\end{aligned}"
    )
)

verified_matrix = interior_matrix.applyfunc(
    lambda entry: reduce_ogmg(entry.subs(closed_form_ogmg))
)
verified_func = interior_func.applyfunc(
    lambda entry: reduce_ogmg(entry.subs(closed_form_ogmg))
)
display_symbolic_residual(
    "OGM-G consecutive-difference residual", verified_matrix, verified_func
)

coefficient matrix rank: 8
coefficient nullity: 2
transition solution before boundary/terminal conditions fix the free constants:


<IPython.core.display.Math object>

closed-form coefficients selected by the boundary/terminal conditions:


<IPython.core.display.Math object>

OGM-G consecutive-difference residual
quadratic residual:


Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0],
[0]])

identity verified? True


### Since $b_k$ is constant, we apply translation and let 
$$\widetilde V_k = \frac{1}{\psi_{k+1}^2}\bigl(f(x_k)-f(x_N)\bigr) - \frac{1}{2L\psi_{k+1}^2} \left\|\nabla f(x_k)\right\|^2 + \frac{\psi_0^2-2\psi_{k+1}^2}{2L\psi_0^2\psi_{k+1}^2}\left\|\nabla f(x_N)\right\|^2 + \frac{L}{\psi_{k+1}^2(2\psi_{k+1}-1)} \langle x_{k+1}-y_{k+1}, z_{k+1}-z_{N+1} \rangle$$
### Following the PEPFlow interpolation convention we let $I(a,b)=f(b)-f(a)+\langle\nabla f(b),a-b\rangle+\frac{1}{2L}\|\nabla f(a)-\nabla f(b)\|^2$
### We prove the two identities: $\widetilde V_N=\widetilde V_{N-1}+I(x_{N-1},x_N)=(\psi_0^2-1)\|\nabla f(x_N)\|^2/(L\psi_0^2)$ (note that $\lambda_{N-1,N} = \psi_N^2 = 1$) and $\widetilde V_0 = \widetilde V_0=\frac{2}{\psi_0^2}(f(x_0)-f(x_N))+\left(\frac{1}{\psi_1^2}-\frac{2}{\psi_0^2}\right)I(x_N,x_0)$.

In [20]:
ctx_ogmg_boundary = pf.PEPContext(
    "ogmg_sparse_boundary_and_terminal_proof"
).set_as_current()

g_0_boundary_abs = pf.Vector(is_basis=True, tags=["g_0"])
g_Nm1_abs = pf.Vector(is_basis=True, tags=["g_{N-1}"])
g_N_boundary_abs = pf.Vector(is_basis=True, tags=["g_N"])
p_Nm1_abs = pf.Vector(is_basis=True, tags=["p_{N-1}"])
x_N_minus_x_0_abs = pf.Vector(is_basis=True, tags=["x_N - x_0"])

f_0_boundary = pf.Scalar(is_basis=True, tags=["f_0"])
f_Nm1_abs = pf.Scalar(is_basis=True, tags=["f_{N-1}"])
f_N_boundary = pf.Scalar(is_basis=True, tags=["f_N"])
f_star_abs = pf.Scalar(is_basis=True, tags=["f_{star}"])

psi_0_boundary_par = pf.Parameter("psi_0")
psi_1_boundary_par = pf.Parameter("psi_1")
half = sp.S(1) / 2

# Last step: psi_N=1, x_N=z_N, and z_{N+1}=x_N-g_N/L.
tilde_V_Nm1_abs = (
    f_Nm1_abs
    - f_N_boundary
    - half / L * g_Nm1_abs**2
    + (psi_0_boundary_par**2 - 2)
    / (2 * L * psi_0_boundary_par**2)
    * g_N_boundary_abs**2
    + p_Nm1_abs * g_N_boundary_abs
)
tilde_V_N_abs = (
    (psi_0_boundary_par**2 - 1) / (L * psi_0_boundary_par**2) * g_N_boundary_abs**2
)

I_Nm1_N_abs = (
    f_N_boundary
    - f_Nm1_abs
    + g_N_boundary_abs * (g_Nm1_abs / L - p_Nm1_abs)
    + half / L * (g_Nm1_abs - g_N_boundary_abs) ** 2
)
tilde_last_step_residual = tilde_V_N_abs - tilde_V_Nm1_abs - I_Nm1_N_abs

# Initial translated value: p_0=x_1-y_1 and v_0=z_1-z_{N+1}.
p_0_abs = -(2 * psi_1_boundary_par - 1) / (L * psi_0_boundary_par) * g_0_boundary_abs
v_0_abs = (
    -x_N_minus_x_0_abs
    - (psi_0_boundary_par + 1) / (2 * L) * g_0_boundary_abs
    + g_N_boundary_abs / L
)
tilde_V_0_sparse_abs = (
    1 / psi_1_boundary_par**2 * (f_0_boundary - f_N_boundary)
    - half / (L * psi_1_boundary_par**2) * g_0_boundary_abs**2
    + (psi_0_boundary_par**2 - 2 * psi_1_boundary_par**2)
    / (2 * L * psi_0_boundary_par**2 * psi_1_boundary_par**2)
    * g_N_boundary_abs**2
    + L / (psi_1_boundary_par**2 * (2 * psi_1_boundary_par - 1)) * (p_0_abs * v_0_abs)
)
I_N_0_abs = (
    f_0_boundary
    - f_N_boundary
    + g_0_boundary_abs * x_N_minus_x_0_abs
    + half / L * (g_N_boundary_abs - g_0_boundary_abs) ** 2
)
tilde_V_0_target_abs = (
    2 / psi_0_boundary_par**2 * (f_0_boundary - f_N_boundary)
    + (1 / psi_1_boundary_par**2 - 2 / psi_0_boundary_par**2) * I_N_0_abs
)
tilde_initial_residual = tilde_V_0_sparse_abs - tilde_V_0_target_abs

# Terminal certificate uses the untranslated partial sum V_N=tilde{V}_N-2 psi_0^{-2}(f_0-f_N).
V_N_abs = tilde_V_N_abs - 2 / psi_0_boundary_par**2 * (f_0_boundary - f_N_boundary)
I_N_star_abs = f_star_abs - f_N_boundary + half / L * g_N_boundary_abs**2
terminal_target = 1 / L * g_N_boundary_abs**2 - 2 / psi_0_boundary_par**2 * (
    f_0_boundary - f_star_abs
)
terminal_residual = V_N_abs + 2 / psi_0_boundary_par**2 * I_N_star_abs - terminal_target

psi_1_sym = sp.Symbol(r"\psi_1")
boundary_resolve = {"L": L_sym, "psi_0": psi_0_sym, "psi_1": psi_1_sym}
boundary_recurrence = psi_0_sym**2 - psi_0_sym - 2 * psi_1_sym**2


def reduce_ogmg_boundary(entry):
    return reduce_by_quadratic_recurrence(entry, psi_0_sym, boundary_recurrence)


def boundary_symbolic_parts(expr):
    matrix_part, func_part = symbolic_parts(expr, ctx_ogmg_boundary, boundary_resolve)
    return (
        matrix_part.applyfunc(reduce_ogmg_boundary),
        func_part.applyfunc(reduce_ogmg_boundary),
    )


last_matrix, last_func = boundary_symbolic_parts(tilde_last_step_residual)
initial_matrix, initial_func = boundary_symbolic_parts(tilde_initial_residual)
terminal_matrix, terminal_func = boundary_symbolic_parts(terminal_residual)

display_symbolic_residual("OGM-G translated last-step residual", last_matrix, last_func)
display_symbolic_residual(
    "OGM-G translated initial-value residual", initial_matrix, initial_func
)
display_symbolic_residual(
    "OGM-G terminal certificate residual", terminal_matrix, terminal_func
)

OGM-G translated last-step residual
quadratic residual:


Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0],
[0]])

identity verified? True
OGM-G translated initial-value residual
quadratic residual:


...

identity verified? True
OGM-G terminal certificate residual
quadratic residual:


Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0],
[0]])

identity verified? True


### Final numerical sanity check: our closed-form formula for $V_k$ agrees with the numerical results

In [21]:
ctx_prf.set_as_current()


def quadratic_from_C(C_i, basis_i):
    quad = pf.Scalar.zero()
    for a in range(len(basis_i)):
        quad += C_i[a, a] * basis_i[a] * basis_i[a]
        for b in range(a + 1, len(basis_i)):
            quad += 2 * C_i[a, b] * basis_i[a] * basis_i[b]
    return quad


def sparse_C_numeric(i: int, N: int, L_value: float = 1.0):
    psi_0_value = float(theta(N, N))
    if i == N:
        return np.array([[(psi_0_value**2 - 1) / (L_value * psi_0_value**2)]])

    if i == N - 1:
        return np.array(
            [
                [-1 / (2 * L_value), 0.0, 0.0],
                [0.0, (psi_0_value**2 - 2) / (2 * L_value * psi_0_value**2), 0.5],
                [0.0, 0.5, 0.0],
            ]
        )

    psi_value = float(theta(N - i - 1, N))
    return np.array(
        [
            [-1 / (2 * L_value * psi_value**2), 0.0, 0.0, 0.0],
            [
                0.0,
                (psi_0_value**2 - 2 * psi_value**2)
                / (2 * L_value * psi_0_value**2 * psi_value**2),
                0.0,
                0.0,
            ],
            [0.0, 0.0, 0.0, L_value / (2 * psi_value**2 * (2 * psi_value - 1))],
            [0.0, 0.0, L_value / (2 * psi_value**2 * (2 * psi_value - 1)), 0.0],
        ]
    )


def sparse_V_formula(i: int):
    x_0 = ctx_prf["x_0"]
    x_N = ctx_prf[f"x_{N_int}"]
    if i == 0:
        psi_value = float(theta(N_int - 1, N_int))
        psi_0_value = float(theta(N_int, N_int))
        p_0 = x[1] - y[1]
        v_0 = z[1] - z[N_int + 1]
        return (
            1 / psi_value**2 * (f(x_0) - f(x_N))
            - 2 / psi_0_value**2 * (f(x_0) - f(x_N))
            - 1 / (2 * float(L_value) * psi_value**2) * g[0] ** 2
            + (psi_0_value**2 - 2 * psi_value**2)
            / (2 * float(L_value) * psi_0_value**2 * psi_value**2)
            * g[N_int] ** 2
            + float(L_value) / (psi_value**2 * (2 * psi_value - 1)) * (p_0 * v_0)
        )

    V = quadratic_from_C(
        sparse_C_numeric(i, N_int, L_value=float(L_value)),
        expected_basis_for_V(i),
    )
    V += -2 / float(theta(N_int, N_int)) ** 2 * (f(x_0) - f(x_N))
    if i < N_int:
        V += (
            1
            / float(theta(N_int - i - 1, N_int)) ** 2
            * (f(ctx_prf[f"x_{i}"]) - f(x_N))
        )
    return V


for i in range(N_int + 1):
    residual = pm.eval_scalar(sparse_V_formula(i) - V_raw[i])
    print(
        f"V_{i} formula residual:",
        "inner=",
        np.max(np.abs(residual.inner_prod_coords.astype(float))),
        "func=",
        np.max(np.abs(residual.func_coords.astype(float))),
    )

V_0 formula residual: inner= 8.215364014074744e-10 func= 3.1680331621930513e-10
V_1 formula residual: inner= 1.1545383121758235e-09 func= 2.6326151814082266e-09
V_2 formula residual: inner= 1.3830225020772247e-09 func= 3.83242562684849e-09
V_3 formula residual: inner= 1.9761429992382062e-09 func= 5.518148882277529e-09
V_4 formula residual: inner= 8.904403547838058e-09 func= 1.5659070984330725e-09
V_5 formula residual: inner= 8.915113314245104e-09 func= 1.5659070984330725e-09
